# Sales Analytics ETL Project

**Tools:** Python, Pandas, Microsoft SQL Server, SQLAlchemy, PyODBC, Matplotlib

## Business Objective

Build an end-to-end ETL pipeline that integrates customer, product, and order datasets and loads a clean sales table into Microsoft SQL Server for business analysis.

The analysis focuses on:
- Total revenue
- Product performance
- Customer spending
- City-level sales
- Monthly trends


## 1. Import Libraries

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import urllib


## 2. Configure File Paths

The notebook is stored in the `notebooks` folder, while the CSV files are stored in `data`.


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

customers_path = DATA_DIR / "customers.csv"
products_path = DATA_DIR / "products.csv"
orders_path = DATA_DIR / "orders.csv"

print("Project root:", PROJECT_ROOT)


## 3. Extract Data

In [ ]:
customers = pd.read_csv(customers_path)
products = pd.read_csv(products_path)
orders = pd.read_csv(orders_path)

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)


In [ ]:
customers.head()

In [ ]:
products.head()

In [ ]:
orders.head()

## 4. Data Quality Checks

In [ ]:
print("Customers missing values:")
print(customers.isnull().sum())

print("\nProducts missing values:")
print(products.isnull().sum())

print("\nOrders missing values:")
print(orders.isnull().sum())

print("\nDuplicate rows:")
print("Customers:", customers.duplicated().sum())
print("Products:", products.duplicated().sum())
print("Orders:", orders.duplicated().sum())


## 5. Transform Data

In [ ]:
customers = customers.drop_duplicates()
products = products.drop_duplicates()
orders = orders.drop_duplicates()

orders["OrderDate"] = pd.to_datetime(orders["OrderDate"], errors="coerce")

sales = orders.merge(
    customers,
    on="CustomerID",
    how="inner"
)

sales = sales.merge(
    products,
    on="ProductID",
    how="inner"
)

sales["TotalAmount"] = sales["Quantity"] * sales["Price"]

sales = sales[
    [
        "OrderID",
        "OrderDate",
        "CustomerID",
        "CustomerName",
        "City",
        "ProductID",
        "ProductName",
        "Category",
        "Quantity",
        "Price",
        "TotalAmount",
    ]
]

sales


## 6. Validate Transformed Data

In [ ]:
print(sales.info())
print("\nMissing values:")
print(sales.isnull().sum())

print("\nDuplicate rows:", sales.duplicated().sum())
print("\nTotal revenue:", sales["TotalAmount"].sum())


## 7. Connect to Microsoft SQL Server

Before running this cell:

1. Create a SQL Server database named `Sales_ETL_DB`.
2. Replace `YOUR_SERVER_NAME` below with your SQL Server instance.
3. Keep Windows Authentication enabled.

Example server names include `localhost\\SQLEXPRESS` or `COMPUTER_NAME\\INSTANCE_NAME`.

> Never upload SQL usernames or passwords to GitHub.


In [ ]:
SERVER_NAME = r"YOUR_SERVER_NAME"
DATABASE_NAME = "Sales_ETL_DB"

connection_string = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)

# Uncomment the following lines after entering your server name.
# with engine.connect() as connection:
#     print("Connected successfully to SQL Server.")


## 8. Load Data into SQL Server

In [ ]:
# Uncomment after configuring the SQL Server connection above.

# sales.to_sql(
#     name="Sales",
#     con=engine,
#     schema="dbo",
#     if_exists="replace",
#     index=False
# )

# print("Sales table loaded successfully.")


## 9. SQL Analysis

After loading the table, uncomment and run the cells below.


In [ ]:
# query = '''
# SELECT *
# FROM dbo.Sales;
# '''
#
# pd.read_sql(query, engine)


### Total Revenue

In [ ]:
# query = '''
# SELECT
#     SUM(TotalAmount) AS TotalRevenue
# FROM dbo.Sales;
# '''
#
# pd.read_sql(query, engine)


### Revenue by Product

In [ ]:
# query = '''
# SELECT
#     ProductName,
#     SUM(TotalAmount) AS Revenue
# FROM dbo.Sales
# GROUP BY ProductName
# ORDER BY Revenue DESC;
# '''
#
# pd.read_sql(query, engine)


### Top Customers

In [ ]:
# query = '''
# SELECT
#     CustomerName,
#     SUM(TotalAmount) AS TotalSpent
# FROM dbo.Sales
# GROUP BY CustomerName
# ORDER BY TotalSpent DESC;
# '''
#
# pd.read_sql(query, engine)


### Revenue by City

In [ ]:
# query = '''
# SELECT
#     City,
#     SUM(TotalAmount) AS Revenue
# FROM dbo.Sales
# GROUP BY City
# ORDER BY Revenue DESC;
# '''
#
# pd.read_sql(query, engine)


### Monthly Revenue

In [ ]:
# query = '''
# SELECT
#     YEAR(OrderDate) AS SalesYear,
#     MONTH(OrderDate) AS SalesMonth,
#     SUM(TotalAmount) AS Revenue
# FROM dbo.Sales
# GROUP BY YEAR(OrderDate), MONTH(OrderDate)
# ORDER BY SalesYear, SalesMonth;
# '''
#
# pd.read_sql(query, engine)


## 10. Python Visualization

In [ ]:
product_revenue = (
    sales.groupby("ProductName", as_index=False)["TotalAmount"]
    .sum()
    .sort_values("TotalAmount", ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(product_revenue["ProductName"], product_revenue["TotalAmount"])
plt.title("Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 11. Business Insights

Based on the sample dataset:

- Product-level revenue analysis highlights the products contributing most to sales.
- Customer-level analysis identifies high-value customers who may be suitable for retention or loyalty initiatives.
- City-level revenue analysis helps compare geographic sales performance.
- Monthly sales analysis can be used to monitor sales trends over time.

## Conclusion

This project demonstrates a basic end-to-end analytics workflow: extracting source data, validating and transforming it with Python, loading it into SQL Server, and analyzing the resulting dataset with SQL and visualizations.
